In [1]:
!pip install -U transformers datasets seqeval pytorch-crf spacy tqdm seaborn
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 25.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
from google.colab import drive
import os
import sys

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!unzip "/content/drive/MyDrive/PV/Test_pv.zip" -d "/content/"

Archive:  /content/drive/MyDrive/PV/Test_pv.zip
replace /content/Test_pv/Configs/model_config.yaml? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: /content/Test_pv/Configs/model_config.yaml  
  inflating: /content/Test_pv/Configs/pipeline_settings.yaml  
  inflating: /content/Test_pv/Configs/stix_mapping.json  
  inflating: /content/Test_pv/Data/Annotations.json  
  inflating: /content/Test_pv/Data/Bosch AnnoCTR/bosch_cti_devtrain_ds.json  
  inflating: /content/Test_pv/Data/Bosch AnnoCTR/bosch_cti_test_ds.json  
  inflating: /content/Test_pv/Data/Bosch AnnoCTR/Generative LLM training instruction sets/sentence_based_train_instructs.json  
  inflating: /content/Test_pv/Data/processed/train_bieos.json  
  inflating: /content/Test_pv/Data/Raw data/ALLANITE.txt  
  inflating: /content/Test_pv/Data/Raw data/Andariel.txt  
  inflating: /content/Test_pv/Data/Raw data/APT1.txt  
  inflating: /content/Test_pv/Data/Raw data/APT12.txt  
  inflating: /content/Test_pv/Data/Raw data/APT18.txt 

In [4]:
project_path = '/content/Test_pv'
os.chdir(project_path)
sys.path.append(project_path)

print(f"✅ Hiện đang ở thư mục: {os.getcwd()}")

✅ Hiện đang ở thư mục: /content/Test_pv


In [5]:
import torch
import yaml
import json
from torch.utils.data import DataLoader, Dataset, random_split
from transformers import  get_linear_schedule_with_warmup
from torch.optim import AdamW
from Joint_model.joint_model import CyberEntRelModel
from Joint_model.tagging_scheme import BIEOSScheme

# Load cấu hình từ file YAML [1]
with open("Configs/model_config.yaml", 'r') as f:
    config = yaml.safe_load(f)

# Khởi tạo sơ đồ nhãn dựa trên mapping [1, 2]
scheme = BIEOSScheme("Configs/stix_mapping.json")
num_labels = scheme.get_num_labels()

# Lấy số lượng loại quan hệ (label_mapping + 1 cho 'no_relation')
with open("Configs/stix_mapping.json", 'r') as f:
    stix_map = json.load(f)
num_rel_types = len(stix_map["label_mapping"]["entities"]) + 1

In [32]:
import torch
from torch.utils.data import Dataset
import json
from transformers import RobertaTokenizerFast

class CTIDataset(Dataset):
    def __init__(self, data_path, model_name="roberta-base"):
        with open(data_path, 'r', encoding='utf-8') as f:
            self.data = json.load(f)

        self.tokenizer = RobertaTokenizerFast.from_pretrained(model_name, add_prefix_space=True)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        tokens = item["tokens"]
        label_ids = item["label_ids"]

        encoding = self.tokenizer(
            tokens,
            is_split_into_words=True,
            truncation=True,
            max_length=self.tokenizer.model_max_length, # Ensure explicit max_length
            return_attention_mask=True,
            return_tensors=None
        )

        # Align word-level labels to token-level inputs
        word_ids = encoding.word_ids()
        token_labels = []
        previous_word_idx = None
        for word_idx in word_ids:
            if word_idx is None: # Special tokens
                token_labels.append(-100)
            elif word_idx != previous_word_idx: # First token of a new word
                token_labels.append(label_ids[word_idx])
            else: # Subsequent tokens of the same word
                token_labels.append(-100)
            previous_word_idx = word_idx

        # Ensure that the length of token_labels matches the length of input_ids
        assert len(token_labels) == len(encoding["input_ids"])

        return {
            "input_ids": torch.tensor(encoding["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(encoding["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(token_labels, dtype=torch.long), # Use the new token_labels
            "rel_labels": torch.tensor(item.get("rel_label", 0), dtype=torch.long)
        }

In [17]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    input_ids = [x["input_ids"] for x in batch]
    attention_mask = [x["attention_mask"] for x in batch]
    labels = [x["labels"] for x in batch]
    rel_labels = torch.tensor([x["rel_labels"] for x in batch], dtype=torch.long)

    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=0)
    attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)
    labels = pad_sequence(labels, batch_first=True, padding_value=-100)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "rel_labels": rel_labels
    }


In [33]:
from torch.utils.data import DataLoader, random_split

full_dataset = CTIDataset("Data/processed/train_bieos.json")

train_size = int(0.8 * len(full_dataset))
val_size = int(0.1 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

print("✅ Dataset sẵn sàng:")
print(f"  - Train: {len(train_dataset)}")
print(f"  - Val:   {len(val_dataset)}")
print(f"  - Test:  {len(test_dataset)}")

✅ Dataset sẵn sàng:
  - Train: 41
  - Val:   5
  - Test:  6


In [34]:
train_loader = DataLoader(
    train_dataset,
    batch_size=config["training"]["batch_size"],
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config["training"]["batch_size"],
    collate_fn=collate_fn
)

In [36]:
import os
import torch
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CyberEntRelModel(
    num_labels=num_labels,
    num_rel_types=num_rel_types,
    model_config=config["model"]
).to(device)

# Optimizer & Scheduler
optimizer = AdamW(
    model.parameters(),
    lr=float(config["training"]["learning_rate"]),
    weight_decay=0.01
)

epochs = 20
total_steps = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

best_val_loss = float("inf")

print(f"🚀 Bắt đầu huấn luyện trên {device}...")

for epoch in range(epochs):
    # ================= TRAIN =================
    model.train()
    total_train_loss = 0.0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        rel_labels = batch["rel_labels"].to(device)

        loss = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            rel_labels=rel_labels
        )

        # Phòng trường hợp loss là tensor batch-wise
        loss = loss.mean()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    # ================= VALIDATION =================
    model.eval()
    total_val_loss = 0.0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            rel_labels = batch["rel_labels"].to(device)

            loss = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
                rel_labels=rel_labels
            )

            total_val_loss += loss.mean().item()

    avg_val_loss = total_val_loss / len(val_loader)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f}"
    )

    # ================= CHECKPOINT =================
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        os.makedirs("models_checkpoints/cyber_joint_v1", exist_ok=True)
        torch.save(
            model.state_dict(),
            "models_checkpoints/cyber_joint_v1/best_model.pt"
        )
        print("💾 Đã lưu mô hình tốt nhất!")

print("🏁 Hoàn thành huấn luyện.")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🚀 Bắt đầu huấn luyện trên cuda...
Epoch 1/20 | Train Loss: 4.2951 | Val Loss: 3.8762
💾 Đã lưu mô hình tốt nhất!
Epoch 2/20 | Train Loss: 4.1265 | Val Loss: 3.6694
💾 Đã lưu mô hình tốt nhất!
Epoch 3/20 | Train Loss: 3.9864 | Val Loss: 3.4891
💾 Đã lưu mô hình tốt nhất!
Epoch 4/20 | Train Loss: 3.8550 | Val Loss: 3.3223
💾 Đã lưu mô hình tốt nhất!
Epoch 5/20 | Train Loss: 3.7350 | Val Loss: 3.1709
💾 Đã lưu mô hình tốt nhất!
Epoch 6/20 | Train Loss: 3.6302 | Val Loss: 3.0310
💾 Đã lưu mô hình tốt nhất!
Epoch 7/20 | Train Loss: 3.5295 | Val Loss: 2.9019
💾 Đã lưu mô hình tốt nhất!
Epoch 8/20 | Train Loss: 3.4384 | Val Loss: 2.7837
💾 Đã lưu mô hình tốt nhất!
Epoch 9/20 | Train Loss: 3.3489 | Val Loss: 2.6758
💾 Đã lưu mô hình tốt nhất!
Epoch 10/20 | Train Loss: 3.2724 | Val Loss: 2.5781
💾 Đã lưu mô hình tốt nhất!
Epoch 11/20 | Train Loss: 3.2030 | Val Loss: 2.4904
💾 Đã lưu mô hình tốt nhất!
Epoch 12/20 | Train Loss: 3.1395 | Val Loss: 2.4125
💾 Đã lưu mô hình tốt nhất!
Epoch 13/20 | Train Loss: 3

In [38]:
from google.colab import files
import os

# Đường dẫn đến file mô hình của bạn
model_path = 'models_checkpoints/cyber_joint_v1/best_model.pt'

if os.path.exists(model_path):
    files.download(model_path)
    print("✅ Đang bắt đầu tải mô hình về máy...")
else:
    print("❌ Không tìm thấy file tại đường dẫn quy định.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Đang bắt đầu tải mô hình về máy...
